# Compile the main simulator code using cython 

In [ ]:
!python ../setup.py build_ext --inplace

# Import all the relevant files 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
import seaborn as sns
import pathos.multiprocessing
import pickle

In [ ]:
import sys
sys.path.append('../')

#Importing scripts:

#Import relevant frames:
import common.cbgt as cbgt
import common.pipeline_creation as pl_creat

#Import plotting functions:
import common.plotting_functions as plt_func
import common.plotting_helper_functions as plt_help
import common.postprocessing_helpers as post_help

importlib.reload(plt_help)
importlib.reload(plt_func)
importlib.reload(post_help)

%load_ext autoreload
%autoreload 2
%reload_ext autoreload 

import warnings
warnings.simplefilter('ignore', category=FutureWarning)

In [ ]:
def saveresults_vars(variable, prefix):
    pickle.dump(variable, open(prefix, 'wb'))
    
def loadresults_vars(prefix):
    return pickle.load(open(prefix, "rb"))

Define data and figures directories and import dataset:

In [ ]:
data_dir = "../Data/"
figure_dir = "../Figures/"

file = data_dir+"PE_summary_75x300_go_stop_plasticityON_timeout1000_uniform"
summary_all = pd.read_pickle(file)

# PES analysis

### To define once: failed stops + plotting functions

In [ ]:
#Identifying failed stops 

failed_stops = (summary_all.query("ttype == 'stop' and reward < 0").loc[:, ["seed", "trials"]].rename(columns={"trials": "t0"}))

MAX_TRIAL = summary_all["trials"].max()

In [ ]:
len(failed_stops)

In [ ]:
def plot_mean_sem(delta_rt_long, figure_dir, fname_suffix):
    """
    delta_rt_long: DataFrame with columns ['seed', 'k', 'delta_RT']
    """

    import numpy as np
    import matplotlib.pyplot as plt

    #Aggregate per seed
    per_seed_pes = (
        delta_rt_long
        .groupby(["seed", "k"])["delta_RT"]
        .mean()
        .reset_index()
    )

    #Wide format for plotting seed traces
    per_seed_wide = (
        per_seed_pes
        .pivot(index="seed", columns="k", values="delta_RT")
        .sort_index(axis=1)
    )

    ks = per_seed_wide.columns.to_numpy(dtype=int)

    mean_pes = per_seed_wide.mean(axis=0).to_numpy(dtype=float)
    sem_pes  = per_seed_wide.sem(axis=0).to_numpy(dtype=float)

    x = ks

    #Plot
    fig, ax = plt.subplots(figsize=(5, 4), tight_layout=True)

    #Mean ± SEM
    ax.errorbar(
        ks, mean_pes, yerr=sem_pes,
        fmt='o-', lw=3, capsize=6,
        color='darkviolet'
    )

    #Individual seed traces
    for _, row in per_seed_wide.iterrows():
        ax.plot(ks, row.values, color='grey', alpha=0.2)

    # Confidence band
    ax.fill_between(
        x,
        mean_pes - sem_pes,
        mean_pes + sem_pes,
        color="#7B2CBF",
        alpha=0.25,
        linewidth=0
    )

    ax.axhline(0, ls='--', c='k', lw=1)

    ax.set_xlabel("Posterror trials", fontsize=15)
    ax.set_ylabel(
        "Post-error ΔRT [ms]\n = post failed k − pre failed",
        fontsize=15
    )

    ax.set_xticks(ks)
    ax.tick_params(axis="both", labelsize=13, width=1)
    ax.spines[['right', 'top']].set_visible(False)

    plt.tight_layout()
    plt.savefig(
        figure_dir + f"meandeltaRT_sem_{fname_suffix}",
        dpi=400
    )
    plt.show()

In [ ]:
def plot_mean_ci(delta_rt_long, figure_dir, fname_suffix, label="Uniform"):
    """
    delta_rt_long: DataFrame with columns ['seed', 'k', 'delta_RT']
    """

    import seaborn as sns
    import matplotlib.pyplot as plt

    pes_long = (
        delta_rt_long
        .rename(columns={"k": "post_trial"})
        .copy()
    )

    plt.figure(figsize=(5, 4), tight_layout=True)
    ax = plt.gca()

    #Vertical guide lines
    for k in range(1, 6):
        ax.axvline(k, color="0.7", lw=1, zorder=0)

    #Mean + CI
    sns.lineplot(
        data=pes_long,
        x="post_trial",
        y="delta_RT",
        estimator="mean",
        ci=95,
        color="slategrey",
        lw=2,
        label=label,
        ax=ax
    )

    ax.axhline(0, ls="--", color="0.4", lw=1.5)

    ax.set_xlim(0.8, 5.2)
    ax.set_xlabel("Posterror trials", fontsize=15)
    ax.set_ylabel("Post-error ΔRT [ms]", fontsize=15)

    ax.tick_params(axis="both", labelsize=15, width=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1)
    ax.spines["bottom"].set_linewidth(1)

    ax.legend(
        loc="upper right",
        bbox_to_anchor=(0.98, 0.98),
        frameon=True,
        fontsize=12
    )

    plt.savefig(
        figure_dir + f"meandeltaRT_ci_{fname_suffix}",
        dpi=400
    )
    plt.show()

# Δ(RT)(k) = RT post k - mean(RT(pre 1, ..., pre 5)) with k \in {1,2,3,4,5}

Pre trials: routine keeps only correct GO trials (failed stop, correct stop, failed go → ignored and excluded from mean computation).  

Post trials: routine computes ΔRT only if post-k is correct GO (other trials are skipped), k remains absolute trial index.

In [ ]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
#RT lookup + trial type lookup + reward lookup
# ---------------------------------------------------------
rt_lookup = summary_all.set_index(["seed", "trials"])["decisionduration"]
ttype_lookup = summary_all.set_index(["seed", "trials"])["ttype"]
reward_lookup = summary_all.set_index(["seed", "trials"])["reward"]

# ---------------------------------------------------------
#Compute ΔRT per k
# ---------------------------------------------------------
delta_rows = []

PRE_WINDOW  = 5
POST_WINDOW = 5

valid_failed = failed_stops.query(
    "t0 >= @PRE_WINDOW and t0 + @POST_WINDOW <= @MAX_TRIAL"
)

for _, row in valid_failed.iterrows():

    seed = row["seed"]
    t0   = row["t0"]

    # -----------------------------------------------------
    # Compute PRE mean (only correct GO trials)
    # -----------------------------------------------------
    pre_rts = []

    for i in range(1, PRE_WINDOW + 1):
        trial_index = t0 - i
        ttype  = ttype_lookup.get((seed, trial_index))
        reward = reward_lookup.get((seed, trial_index))
        rt     = rt_lookup.get((seed, trial_index), np.nan)

        # Correct GO only
        if ttype == "go" and reward > 0 and not np.isnan(rt):
            pre_rts.append(rt)

    if len(pre_rts) == 0:
        continue  # cannot compute baseline

    pre_mean = np.mean(pre_rts)

    # -----------------------------------------------------
    # Compute post-k ΔRT (only correct GO trials)
    # -----------------------------------------------------
    for k in range(1, POST_WINDOW + 1):

        trial_index = t0 + k
        ttype  = ttype_lookup.get((seed, trial_index))
        reward = reward_lookup.get((seed, trial_index))
        rt     = rt_lookup.get((seed, trial_index), np.nan)

        # Post trial must be correct GO
        if not (ttype == "go" and reward > 0 and not np.isnan(rt)):
            continue

        delta_rows.append({
            "seed": seed,
            "t0": t0,
            "k": k,
            "delta_RT": rt - pre_mean
        })

# ---------------------------------------------------------
# Final long-format dataframe
# ---------------------------------------------------------
delta_rt_R_preMean = pd.DataFrame(delta_rows)

delta_rt_R_preMean["delta_RT"] = pd.to_numeric(
    delta_rt_R_preMean["delta_RT"], errors="coerce"
)
delta_rt_R_preMean["delta_RT"] = delta_rt_R_preMean["delta_RT"].astype(float)

In [ ]:
# Ns per k
delta_rt_R_preMean.groupby("k").size()

In [ ]:
plot_mean_sem(delta_rt_R_preMean, figure_dir, "PEslowing")

In [ ]:
plot_mean_ci(delta_rt_R_preMean, figure_dir, "PEslowing")